In [0]:
# BRONZE LAYER: Ingestao de dados JSON das Copas do Mundo (1930-2022)
# Download via API GitHub e persistencia em Unity Catalog Volume

import requests
import json
import os
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql import functions as F

print("IMPORTACAO: Todos JSONs do worldcup.json")
print("="*80)

print("\nJSONs disponiveis no repositorio worldcup.json:")

# Anos das copas disponíveis no repositório
anos_copas = [
    '1930', '1934', '1938', '1950', '1954', '1958', '1962', '1966', '1970',
    '1974', '1978', '1982', '1986', '1990', '1994', '1998', '2002', '2006',
    '2010', '2014', '2018', '2022'
]

base_url = "https://raw.githubusercontent.com/openfootball/worldcup.json/master"

# Criar schema Bronze e Volume Unity Catalog
print("\nCriando schema Bronze e Volume...")
spark.sql("CREATE DATABASE IF NOT EXISTS workspace.bronze")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.bronze.football_raw")
print("Schema workspace.bronze criado/verificado")
print("Volume workspace.bronze.football_raw criado/verificado")

volume_json_path = "/Volumes/workspace/bronze/football_raw/json"
os.makedirs(volume_json_path, exist_ok=True)
print(f"Diretorio criado: {volume_json_path}")

# Download de JSONs do GitHub e persistencia no volume
print("\nBaixando e salvando JSONs no volume...")

registros = []
partida_counter = 1
jsons_baixados = 0
jsons_com_erro = []

for ano in anos_copas:
    url = f"{base_url}/{ano}/worldcup.json"
    arquivo_destino = f"{volume_json_path}/worldcup_{ano}.json"
    
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        worldcup_data = response.json()
        
        # Salvar JSON no volume
        with open(arquivo_destino, 'w', encoding='utf-8') as f:
            json.dump(worldcup_data, f, ensure_ascii=False, indent=2)
        
        copa_nome = worldcup_data.get('name', f'Copa {ano}')
        total_partidas = len(worldcup_data.get('matches', []))
        
        print(f"  {ano}: {copa_nome} - {total_partidas} partidas -> worldcup_{ano}.json")
        jsons_baixados += 1
        
        # Parse de cada partida: extrair metadados, placares e gols
        for partida in worldcup_data.get('matches', []):
            partida_id = f"{ano}_{partida_counter}"
            round_name = partida.get('round', '')
            match_date = partida.get('date', '')
            match_time = partida.get('time', '')
            team1 = partida.get('team1', '')
            team2 = partida.get('team2', '')
            group = partida.get('group', '')
            ground = partida.get('ground', '')
            
            # Extrair placares (full-time e half-time)
            score = partida.get('score', {})
            score_ft = score.get('ft', [None, None])
            score_ht = score.get('ht', [None, None])
            
            score_ft_home = score_ft[0] if len(score_ft) > 0 else None
            score_ft_away = score_ft[1] if len(score_ft) > 1 else None
            score_ht_home = score_ht[0] if len(score_ht) > 0 else None
            score_ht_away = score_ht[1] if len(score_ht) > 1 else None
            
            # Extrair listas de gols e serializar como JSON string para armazenamento
            goals1 = partida.get('goals1', [])
            goals2 = partida.get('goals2', [])
            goals1_json = json.dumps(goals1, ensure_ascii=False)
            goals2_json = json.dumps(goals2, ensure_ascii=False)
            
            registro = {
                'partida_id': partida_id,
                'ano_copa': ano,
                'copa_nome': copa_nome,
                'round': round_name,
                'match_date': match_date,
                'match_time': match_time,
                'team_home': team1,
                'team_away': team2,
                'score_ft_home': score_ft_home,
                'score_ft_away': score_ft_away,
                'score_ht_home': score_ht_home,
                'score_ht_away': score_ht_away,
                'group': group,
                'ground': ground,
                'goals_home': goals1_json,
                'goals_away': goals2_json
            }
            
            registros.append(registro)
            partida_counter += 1
            
    except Exception as e:
        jsons_com_erro.append(f"{ano}: {str(e)}")
        print(f"  ERRO {ano}: {str(e)}")

print(f"\n{jsons_baixados} JSONs baixados e salvos no volume com sucesso")
if jsons_com_erro:
    print(f"{len(jsons_com_erro)} JSONs com erro")
print(f"{len(registros)} partidas processadas")

print(f"\nArquivos no volume:")
arquivos = sorted([f for f in os.listdir(volume_json_path) if f.endswith('.json')])
print(f"  Total: {len(arquivos)} arquivos JSON")
for arq in arquivos[:5]:
    tamanho = os.path.getsize(f"{volume_json_path}/{arq}") / 1024
    print(f"    {arq} ({tamanho:.1f} KB)")
if len(arquivos) > 5:
    print(f"    ... e mais {len(arquivos) - 5} arquivos")

# Criar DataFrame Spark com schema explícito
print("\nCriando DataFrame Spark...")

schema = StructType([
    StructField("partida_id", StringType(), True),
    StructField("ano_copa", StringType(), True),
    StructField("copa_nome", StringType(), True),
    StructField("round", StringType(), True),
    StructField("match_date", StringType(), True),
    StructField("match_time", StringType(), True),
    StructField("team_home", StringType(), True),
    StructField("team_away", StringType(), True),
    StructField("score_ft_home", IntegerType(), True),
    StructField("score_ft_away", IntegerType(), True),
    StructField("score_ht_home", IntegerType(), True),
    StructField("score_ht_away", IntegerType(), True),
    StructField("group", StringType(), True),
    StructField("ground", StringType(), True),
    StructField("goals_home", StringType(), True),
    StructField("goals_away", StringType(), True)
])

df_matches = spark.createDataFrame([Row(**r) for r in registros], schema=schema)
print(f"DataFrame criado com {df_matches.count()} registros")

# Salvar como tabela Delta no catalog
print("\nSalvando tabela Delta...")

df_matches.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.bronze.matches_json")

print("Tabela workspace.bronze.matches_json criada com sucesso!")

# Validacao e estatisticas
print("\nValidacao da tabela:")
total = spark.table('workspace.bronze.matches_json').count()
print(f"  Total de registros: {total}")

print("\nSchema da tabela:")
spark.table('workspace.bronze.matches_json').printSchema()

print("\nEstatisticas por Copa:")
spark.sql("""
    SELECT 
        ano_copa,
        copa_nome,
        COUNT(*) as total_partidas,
        SUM(score_ft_home) + SUM(score_ft_away) as total_gols
    FROM workspace.bronze.matches_json
    GROUP BY ano_copa, copa_nome
    ORDER BY ano_copa
""").show(100, truncate=False)

print("\nEstatisticas Gerais:")
spark.sql("""
    SELECT 
        COUNT(DISTINCT ano_copa) as total_copas,
        COUNT(*) as total_partidas,
        COUNT(DISTINCT team_home) + COUNT(DISTINCT team_away) as total_times,
        SUM(score_ft_home) + SUM(score_ft_away) as total_gols,
        COUNT(DISTINCT ground) as total_estadios
    FROM workspace.bronze.matches_json
""").show(truncate=False)

print("\nPreview (primeiras 10 partidas):")
display(spark.table('workspace.bronze.matches_json').limit(10))

print("\nImportacao concluida!")



In [0]:
# BRONZE LAYER: Explosao dos gols em registros individuais
# Transforma arrays JSON de gols em linhas separadas (1 linha = 1 gol)

from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StructType, StructField, StringType

print("CRIACAO: Tabela matches_goals")
print("="*80)

df_matches = spark.table('workspace.bronze.matches_json')
print(f"Lendo tabela matches_json: {df_matches.count()} partidas")

# Processar gols do time da casa
print("\nProcessando gols do time da casa...")

df_goals_home = df_matches \
    .select(
        'partida_id',
        'ano_copa',
        'team_home',
        'goals_home'
    ) \
    .filter(F.col('goals_home') != '[]') \
    .withColumn('goals_array', F.from_json(F.col('goals_home'), ArrayType(StructType([
        StructField('name', StringType(), True),
        StructField('minute', StringType(), True),
        StructField('score', ArrayType(StringType()), True),
        StructField('penalty', StringType(), True),
        StructField('owngoal', StringType(), True),
        StructField('offset', StringType(), True)
    ])))) \
    .withColumn('gol', F.explode('goals_array')) \
    .select(
        'partida_id',
        'ano_copa',
        F.lit('home').alias('type'),
        F.col('team_home').alias('team'),
        F.col('gol.name').alias('nome_jogador'),
        F.col('gol.minute').alias('tempo_gol'),
        F.col('gol.penalty').alias('penalty'),
        F.col('gol.owngoal').alias('owngoal')
    )

print(f"  {df_goals_home.count()} gols do time da casa processados")

# Processar gols do time visitante
print("\nProcessando gols do time visitante...")

df_goals_away = df_matches \
    .select(
        'partida_id',
        'ano_copa',
        'team_away',
        'goals_away'
    ) \
    .filter(F.col('goals_away') != '[]') \
    .withColumn('goals_array', F.from_json(F.col('goals_away'), ArrayType(StructType([
        StructField('name', StringType(), True),
        StructField('minute', StringType(), True),
        StructField('score', ArrayType(StringType()), True),
        StructField('penalty', StringType(), True),
        StructField('owngoal', StringType(), True),
        StructField('offset', StringType(), True)
    ])))) \
    .withColumn('gol', F.explode('goals_array')) \
    .select(
        'partida_id',
        'ano_copa',
        F.lit('away').alias('type'),
        F.col('team_away').alias('team'),
        F.col('gol.name').alias('nome_jogador'),
        F.col('gol.minute').alias('tempo_gol'),
        F.col('gol.penalty').alias('penalty'),
        F.col('gol.owngoal').alias('owngoal')
    )

print(f"  {df_goals_away.count()} gols do time visitante processados")

# Union de gols casa + visitante
print("\nUnindo gols de ambos os times...")

df_goals = df_goals_home.union(df_goals_away)
print(f"  {df_goals.count()} gols totais")

# Ordenar cronologicamente por partida e minuto do gol
print("\nOrdenando por partida_id e tempo_gol...")
df_goals = df_goals.orderBy(
    F.col('partida_id').asc(),
    F.col('tempo_gol').cast('int').asc()
)
print("  Dados ordenados")

print("\nSalvando tabela Delta...")

df_goals.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.bronze.matches_goals")

print("Tabela workspace.bronze.matches_goals criada com sucesso!")

# Validacao e estatisticas
print("\nValidacao da tabela:")
total_gols = spark.table('workspace.bronze.matches_goals').count()
print(f"  Total de gols: {total_gols}")

print("\nSchema da tabela:")
spark.table('workspace.bronze.matches_goals').printSchema()

print("\nEstatisticas:")
spark.sql("""
    SELECT 
        COUNT(DISTINCT partida_id) as partidas_com_gols,
        COUNT(DISTINCT nome_jogador) as jogadores_artilheiros,
        COUNT(DISTINCT team) as times,
        COUNT(*) as total_gols,
        SUM(CASE WHEN penalty = 'true' THEN 1 ELSE 0 END) as gols_penalti,
        SUM(CASE WHEN owngoal = 'true' THEN 1 ELSE 0 END) as gols_contra
    FROM workspace.bronze.matches_goals
""").show(truncate=False)

print("\nGols por tipo:")
spark.sql("""
    SELECT 
        type,
        COUNT(*) as total_gols
    FROM workspace.bronze.matches_goals
    GROUP BY type
    ORDER BY type
""").show(truncate=False)

print("\nTop 10 artilheiros:")
spark.sql("""
    SELECT 
        nome_jogador,
        team,
        COUNT(*) as gols
    FROM workspace.bronze.matches_goals
    GROUP BY nome_jogador, team
    ORDER BY gols DESC
    LIMIT 10
""").show(truncate=False)

print("\nPreview (primeiros 20 gols):")
display(spark.table('workspace.bronze.matches_goals').limit(20))

print("\nTabela matches_goals criada com sucesso!")

In [0]:
# SILVER LAYER: Join de partidas e gols
# Combina metadados das partidas com detalhes individuais de cada gol

from pyspark.sql import functions as F

print("CRIACAO: Database Silver e tabela matches_silver")
print("="*80)

print("\nCriando database Silver...")
spark.sql("CREATE DATABASE IF NOT EXISTS workspace.silver")
print("Database workspace.silver criado/verificado")

print("\nLendo tabelas Bronze...")
df_matches_json = spark.table('workspace.bronze.matches_json')
df_matches_goals = spark.table('workspace.bronze.matches_goals')

print(f"  matches_json: {df_matches_json.count()} partidas")
print(f"  matches_goals: {df_matches_goals.count()} gols")

# LEFT JOIN: preserva partidas sem gols (0x0, desistencias, etc)
print("\nFazendo join das tabelas...")

# Evitar duplicacao da coluna ano_copa no join
df_goals_selected = df_matches_goals.select(
    'partida_id',
    F.col('type').alias('gol_type'),
    F.col('team').alias('gol_team'),
    'nome_jogador',
    'tempo_gol',
    'penalty',
    'owngoal'
)

# LEFT JOIN: todas as partidas + gols (quando existirem)
df_silver = df_matches_json.join(
    df_goals_selected,
    on='partida_id',
    how='left'
)

print(f"  Join realizado: {df_silver.count()} registros")

print("\nSalvando tabela Delta no schema Silver...")

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.matches_silver")

print("Tabela workspace.silver.matches_silver criada com sucesso!")

# Validacao
print("\nValidacao da tabela:")
total = spark.table('workspace.silver.matches_silver').count()
print(f"  Total de registros: {total}")

print("\nSchema da tabela:")
spark.table('workspace.silver.matches_silver').printSchema()

print("\nEstatisticas:")
spark.sql("""
    SELECT 
        COUNT(DISTINCT partida_id) as total_partidas,
        COUNT(DISTINCT ano_copa) as total_copas,
        COUNT(nome_jogador) as registros_com_gols,
        COUNT(DISTINCT nome_jogador) as jogadores_unicos,
        COUNT(DISTINCT CONCAT(team_home, ' x ', team_away)) as confrontos_unicos
    FROM workspace.silver.matches_silver
""").show(truncate=False)

print("\nPartidas com e sem gols:")
spark.sql("""
    SELECT 
        CASE 
            WHEN nome_jogador IS NULL THEN 'Sem gols registrados'
            ELSE 'Com gols registrados'
        END as status,
        COUNT(DISTINCT partida_id) as total_partidas
    FROM workspace.silver.matches_silver
    GROUP BY status
    ORDER BY status
""").show(truncate=False)

print("\nPreview - Partida com gols (primeiros 10 registros):")
spark.sql("""
    SELECT 
        partida_id,
        ano_copa,
        team_home,
        team_away,
        score_ft_home,
        score_ft_away,
        gol_type,
        gol_team,
        nome_jogador,
        tempo_gol
    FROM workspace.silver.matches_silver
    WHERE nome_jogador IS NOT NULL
    ORDER BY partida_id, tempo_gol
    LIMIT 10
""").show(truncate=False)

print("\nPreview - Partida sem gols (primeiras 5):")
spark.sql("""
    SELECT 
        partida_id,
        ano_copa,
        team_home,
        team_away,
        score_ft_home,
        score_ft_away,
        nome_jogador
    FROM workspace.silver.matches_silver
    WHERE nome_jogador IS NULL
    LIMIT 5
""").show(truncate=False)

print("\nTabela Silver criada com sucesso!")

In [0]:
# SILVER LAYER: Limpeza e ordenacao
# Remove colunas redundantes e ordena dados cronologicamente

from pyspark.sql import functions as F

print("TRATAMENTO: Tabela matches_silver")
print("="*80)

print("\nLendo tabela matches_silver...")
df_silver = spark.table('workspace.silver.matches_silver')
print(f"  {df_silver.count()} registros")

print("\nColunas originais:")
print(f"  Total: {len(df_silver.columns)} colunas")
for col in df_silver.columns:
    print(f"    {col}")

# Remover colunas goals_home e goals_away (dados ja expandidos em matches_goals)
print("\nRemovendo colunas redundantes (goals_home, goals_away)...")
df_silver_tratado = df_silver.drop('goals_home', 'goals_away')
print(f"  Colunas removidas")

print("\nColunas apos remocao:")
print(f"  Total: {len(df_silver_tratado.columns)} colunas")
for col in df_silver_tratado.columns:
    print(f"    {col}")

# Ordenar cronologicamente: primeiro por data, depois por minuto do gol
print("\nOrdenando por match_date e tempo_gol...")
df_silver_tratado = df_silver_tratado.orderBy(
    F.col('match_date').asc(),
    F.col('tempo_gol').cast('int').asc()
)
print("  Dados ordenados")

print("\nSalvando tabela tratada...")

df_silver_tratado.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.matches_silver")

print("Tabela workspace.silver.matches_silver atualizada com sucesso!")

# Validacao
print("\nValidacao da tabela tratada:")
total = spark.table('workspace.silver.matches_silver').count()
print(f"  Total de registros: {total}")

print("\nSchema final:")
spark.table('workspace.silver.matches_silver').printSchema()

print("\nPreview ordenado (primeiros 15 registros):")
spark.sql("""
    SELECT 
        partida_id,
        ano_copa,
        team_home,
        team_away,
        score_ft_home,
        score_ft_away,
        gol_type,
        gol_team,
        nome_jogador,
        tempo_gol
    FROM workspace.silver.matches_silver
    ORDER BY partida_id ASC, tempo_gol ASC
    LIMIT 15
""").show(truncate=False)

print("\nTratamento Silver concluido!")

In [0]:
# SILVER LAYER: Normalizacao de valores vazios
# Converte strings vazias para NULL para consistencia de dados

from pyspark.sql import functions as F

print("LIMPEZA: Substituir dados em branco por NULL")
print("="*80)

print("\nLendo tabela matches_silver...")
df_silver = spark.table('workspace.silver.matches_silver')
print(f"  {df_silver.count()} registros")

print("\nIdentificando colunas string...")
string_columns = [field.name for field in df_silver.schema.fields if str(field.dataType) == 'StringType()']
print(f"  {len(string_columns)} colunas string encontradas")
for col in string_columns:
    print(f"    {col}")

# Aplicar transformacao em batch usando withColumns (mais eficiente que loop)
print("\nSubstituindo strings vazias por NULL...")

# Criar dicionario de transformacoes para todas as colunas string
transformations = {
    col_name: F.when(F.col(col_name) == '', None).otherwise(F.col(col_name))
    for col_name in string_columns
}

# Aplicar todas as transformacoes de uma so vez
df_silver_clean = df_silver.withColumns(transformations)

print("  Substituicao concluida")

print("\nAnalise de valores vazios:")
print("\nAntes da limpeza:")
for col_name in string_columns[:5]:  # Mostra primeiras 5 colunas
    count_empty = df_silver.filter(F.col(col_name) == '').count()
    if count_empty > 0:
        print(f"  {col_name}: {count_empty} valores vazios")

print("\nDepois da limpeza:")
for col_name in string_columns[:5]:  # Mostra primeiras 5 colunas
    count_empty = df_silver_clean.filter(F.col(col_name) == '').count()
    count_null = df_silver_clean.filter(F.col(col_name).isNull()).count()
    print(f"  {col_name}: {count_empty} vazios, {count_null} NULL")

print("\nSalvando tabela limpa...")

df_silver_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("mergeSchema", "true") \
    .saveAsTable("workspace.silver.matches_silver")

print("Tabela workspace.silver.matches_silver atualizada com sucesso!")

print("\nValidacao final:")
total = spark.table('workspace.silver.matches_silver').count()
print(f"  Total de registros: {total}")

print("\nPreview (primeiros 10 registros):")
spark.sql("""
    SELECT 
        partida_id,
        ano_copa,
        team_home,
        team_away,
        match_time,
        gol_type,
        nome_jogador,
        tempo_gol
    FROM workspace.silver.matches_silver
    LIMIT 10
""").show(truncate=False)

print("\nLimpeza concluida! Strings vazias convertidas para NULL.")

In [0]:
# SILVER LAYER: Enriquecimento e tipagem
# Conversao de tipos (string -> date/int/bool) e criacao de campos calculados

from pyspark.sql import functions as F
from pyspark.sql.types import DateType, IntegerType, BooleanType

print("PREPARACAO FINAL: Conversoes e Enriquecimento")
print("="*80)

print("\nLendo tabela matches_silver...")
df_silver = spark.table('workspace.silver.matches_silver')
print(f"  {df_silver.count()} registros")
print(f"  {len(df_silver.columns)} colunas")

# Conversao de tipos para otimizar queries e permitir calculos
print("\nAplicando conversoes de tipos...")

df_prepared = df_silver \
    .withColumn('match_date', F.to_date(F.col('match_date'), 'yyyy-MM-dd')) \
    .withColumn('tempo_gol', F.col('tempo_gol').cast(IntegerType())) \
    .withColumn('penalty', F.when(F.col('penalty') == 'true', True).otherwise(False).cast(BooleanType())) \
    .withColumn('owngoal', F.when(F.col('owngoal') == 'true', True).otherwise(False).cast(BooleanType()))

print("  match_date -> DATE")
print("  tempo_gol -> INTEGER")
print("  penalty -> BOOLEAN")
print("  owngoal -> BOOLEAN")

# Campos calculados para facilitar analises
print("\nAdicionando campos calculados...")

df_prepared = df_prepared \
    .withColumn('total_gols_partida', F.col('score_ft_home') + F.col('score_ft_away')) \
    .withColumn('resultado_partida',
        F.when(F.col('score_ft_home') > F.col('score_ft_away'), 'Vitória Casa')
         .when(F.col('score_ft_home') < F.col('score_ft_away'), 'Vitória Visitante')
         .otherwise('Empate')
    ) \
    .withColumn('periodo_gol',
        F.when(F.col('tempo_gol').isNull(), None)
         .when(F.col('tempo_gol').between(1, 45), '1º Tempo')
         .when(F.col('tempo_gol').between(46, 90), '2º Tempo')
         .when(F.col('tempo_gol') > 90, 'Prorrogação')
         .otherwise('Desconhecido')
    )

print("  total_gols_partida: soma dos gols da partida")
print("  resultado_partida: Vitoria Casa / Empate / Vitoria Visitante")
print("  periodo_gol: 1o Tempo / 2o Tempo / Prorrogacao")

# Flags booleanas para filtros rapidos
print("\nAdicionando flags analiticas...")

df_prepared = df_prepared \
    .withColumn('foi_goleada', 
        (F.abs(F.col('score_ft_home') - F.col('score_ft_away')) >= 3).cast(BooleanType())
    )

print("  foi_goleada: diferenca >= 3 gols")

# Extrair componentes de data para analises temporais
print("\nExtraindo componentes temporais...")

df_prepared = df_prepared \
    .withColumn('ano_partida', F.year(F.col('match_date'))) \
    .withColumn('mes_partida', F.month(F.col('match_date'))) \
    .withColumn('decada', 
        F.concat(
            F.floor(F.col('ano_copa').cast(IntegerType()) / 10) * 10,
            F.lit('s')
        )
    )

print("  ano_partida: ano extraido de match_date")
print("  mes_partida: mes extraido de match_date")
print("  decada: decada da copa (1930s, 1940s, etc.)")

print("\nSalvando tabela preparada...")

df_prepared.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.matches_silver")

print("Tabela workspace.silver.matches_silver atualizada com sucesso!")

# Validacao e estatisticas
print("\nValidacao final:")
df_final = spark.table('workspace.silver.matches_silver')
total = df_final.count()
print(f"  Total de registros: {total}")
print(f"  Total de colunas: {len(df_final.columns)}")

print("\nSchema atualizado:")
df_final.printSchema()

print("\nEstatisticas de conversoes:")
stats_df = df_final.agg(
    F.count('*').alias('total_registros'),
    F.count('match_date').alias('datas_validas'),
    F.count('tempo_gol').alias('gols_com_tempo'),
    F.sum(F.col('penalty').cast('int')).alias('penaltis'),
    F.sum(F.col('owngoal').cast('int')).alias('gols_contra'),
    F.sum(F.col('foi_goleada').cast('int')).alias('goleadas')
)
stats_df.show(truncate=False)

print("\nDistribuicao por resultado:")
spark.sql("""
    SELECT 
        resultado_partida,
        COUNT(DISTINCT partida_id) as total_partidas,
        ROUND(COUNT(DISTINCT partida_id) * 100.0 / (SELECT COUNT(DISTINCT partida_id) FROM workspace.silver.matches_silver), 2) as percentual
    FROM workspace.silver.matches_silver
    GROUP BY resultado_partida
    ORDER BY total_partidas DESC
""").show(truncate=False)

print("\nDistribuicao por periodo de gol:")
spark.sql("""
    SELECT 
        periodo_gol,
        COUNT(*) as total_gols,
        ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM workspace.silver.matches_silver WHERE periodo_gol IS NOT NULL), 2) as percentual
    FROM workspace.silver.matches_silver
    WHERE periodo_gol IS NOT NULL
    GROUP BY periodo_gol
    ORDER BY 
        CASE periodo_gol
            WHEN '1º Tempo' THEN 1
            WHEN '2º Tempo' THEN 2
            WHEN 'Prorrogação' THEN 3
            ELSE 4
        END
""").show(truncate=False)

print("\nEstatisticas por decada:")
spark.sql("""
    SELECT 
        decada,
        COUNT(DISTINCT partida_id) as total_partidas,
        SUM(total_gols_partida) as total_gols,
        ROUND(AVG(total_gols_partida), 2) as media_gols_partida
    FROM workspace.silver.matches_silver
    GROUP BY decada
    ORDER BY decada
""").show(truncate=False)

print("\nPreview dos dados preparados (5 registros com gols):")
spark.sql("""
    SELECT 
        partida_id,
        match_date,
        team_home,
        team_away,
        resultado_partida,
        total_gols_partida,
        nome_jogador,
        tempo_gol,
        periodo_gol,
        penalty,
        owngoal,
        foi_goleada,
        decada
    FROM workspace.silver.matches_silver
    WHERE nome_jogador IS NOT NULL
    ORDER BY match_date DESC
    LIMIT 5
""").show(truncate=False)

print("\nPreparacao concluida! Dados prontos para a camada Gold.")
print("\nProximos passos:")
print("  1. Criar tabelas dimensionais (dim_times, dim_copas, dim_estadios)")
print("  2. Criar tabelas fato (fato_partidas, fato_gols)")
print("  3. Criar views analiticas agregadas")

In [0]:
# GOLD LAYER: Modelo dimensional Star Schema
# 5 dimensoes (copas, times, estadios, jogadores, tempo) + 2 fatos (partidas, gols)
# Otimizado para queries analiticas e BI

from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("CRIACAO: Database Gold - Modelo Dimensional")
print("="*80)

print("\nCriando database Gold...")
spark.sql("CREATE DATABASE IF NOT EXISTS workspace.gold")
print("Database workspace.gold criado/verificado")

print("\nLendo tabela Silver preparada...")
df_silver = spark.table('workspace.silver.matches_silver')
print(f"  {df_silver.count()} registros")
print(f"  {len(df_silver.columns)} colunas")

# Criacao de dimensoes: tabelas lookup com atributos descritivos
print("\n" + "="*80)
print("CRIANDO DIMENSAO: dim_copas")
print("="*80)

dim_copas = df_silver \
    .select('ano_copa', 'copa_nome', 'decada') \
    .distinct() \
    .withColumn('copa_id', F.monotonically_increasing_id()) \
    .select('copa_id', 'ano_copa', 'copa_nome', 'decada') \
    .orderBy('ano_copa')

print(f"  {dim_copas.count()} copas distintas")

dim_copas.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.dim_copas")

print("Tabela workspace.gold.dim_copas criada!")

print("\n" + "="*80)
print("CRIANDO DIMENSAO: dim_times")
print("="*80)

# Union de times casa + visitante para lista completa
team_home = df_silver.select(F.col('team_home').alias('time_nome')).distinct()
team_away = df_silver.select(F.col('team_away').alias('time_nome')).distinct()

dim_times = team_home.union(team_away).distinct() \
    .withColumn('time_id', F.monotonically_increasing_id()) \
    .select('time_id', 'time_nome') \
    .orderBy('time_nome')

print(f"  {dim_times.count()} times distintos")

dim_times.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.dim_times")

print("Tabela workspace.gold.dim_times criada!")

print("\n" + "="*80)
print("CRIANDO DIMENSAO: dim_estadios")
print("="*80)

dim_estadios = df_silver \
    .select('ground') \
    .filter(F.col('ground').isNotNull()) \
    .distinct() \
    .withColumn('estadio_id', F.monotonically_increasing_id()) \
    .withColumnRenamed('ground', 'estadio_nome') \
    .select('estadio_id', 'estadio_nome') \
    .orderBy('estadio_nome')

print(f"  {dim_estadios.count()} estadios distintos")

dim_estadios.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.dim_estadios")

print("Tabela workspace.gold.dim_estadios criada!")

print("\n" + "="*80)
print("CRIANDO DIMENSAO: dim_jogadores")
print("="*80)

dim_jogadores = df_silver \
    .select('nome_jogador', 'gol_team') \
    .filter(F.col('nome_jogador').isNotNull()) \
    .distinct() \
    .withColumn('jogador_id', F.monotonically_increasing_id()) \
    .withColumnRenamed('gol_team', 'time_principal') \
    .select('jogador_id', 'nome_jogador', 'time_principal') \
    .orderBy('nome_jogador')

print(f"  {dim_jogadores.count()} jogadores distintos")

dim_jogadores.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.dim_jogadores")

print("Tabela workspace.gold.dim_jogadores criada!")

print("\n" + "="*80)
print("CRIANDO DIMENSAO: dim_tempo")
print("="*80)

dim_tempo = df_silver \
    .select('match_date', 'ano_partida', 'mes_partida') \
    .distinct() \
    .withColumn('tempo_id', F.monotonically_increasing_id()) \
    .withColumn('dia_partida', F.dayofmonth('match_date')) \
    .withColumn('dia_semana', F.dayofweek('match_date')) \
    .withColumn('trimestre', F.quarter('match_date')) \
    .select(
        'tempo_id',
        'match_date',
        'ano_partida',
        'mes_partida',
        'dia_partida',
        'dia_semana',
        'trimestre'
    ) \
    .orderBy('match_date')

print(f"  {dim_tempo.count()} datas distintas")

dim_tempo.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.dim_tempo")

print("Tabela workspace.gold.dim_tempo criada!")

# Criacao de fatos: tabelas de metricas com foreign keys para dimensoes
print("\n" + "="*80)
print("CRIANDO FATO: fato_partidas")
print("="*80)

# Carregar dimensoes para fazer joins com foreign keys
df_copas = spark.table('workspace.gold.dim_copas')
df_times = spark.table('workspace.gold.dim_times')
df_estadios = spark.table('workspace.gold.dim_estadios')
df_tempo = spark.table('workspace.gold.dim_tempo')

# Fato_partidas: granularidade = 1 partida (965 linhas)
fato_partidas = df_silver \
    .select(
        'partida_id',
        'ano_copa',
        'match_date',
        'team_home',
        'team_away',
        'ground',
        'round',
        'group',
        'score_ft_home',
        'score_ft_away',
        'score_ht_home',
        'score_ht_away',
        'total_gols_partida',
        'resultado_partida',
        'foi_goleada'
    ) \
    .distinct()

# Joins para substituir atributos descritivos por IDs (normalizacao)
fato_partidas = fato_partidas \
    .join(df_copas, on='ano_copa', how='left') \
    .join(df_times.withColumnRenamed('time_id', 'time_casa_id').withColumnRenamed('time_nome', 'team_home'), on='team_home', how='left') \
    .join(df_times.withColumnRenamed('time_id', 'time_visitante_id').withColumnRenamed('time_nome', 'team_away'), on='team_away', how='left') \
    .join(df_estadios.withColumnRenamed('estadio_nome', 'ground'), on='ground', how='left') \
    .join(df_tempo, on='match_date', how='left') \
    .select(
        'partida_id',
        'copa_id',
        'tempo_id',
        'time_casa_id',
        'time_visitante_id',
        'estadio_id',
        'round',
        'group',
        'score_ft_home',
        'score_ft_away',
        'score_ht_home',
        'score_ht_away',
        'total_gols_partida',
        'resultado_partida',
        'foi_goleada'
    )

print(f"  {fato_partidas.count()} partidas")

fato_partidas.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.fato_partidas")

print("Tabela workspace.gold.fato_partidas criada!")

print("\n" + "="*80)
print("CRIANDO FATO: fato_gols")
print("="*80)

df_jogadores = spark.table('workspace.gold.dim_jogadores')

# Fato_gols: granularidade = 1 gol (841 linhas)
fato_gols = df_silver \
    .filter(F.col('nome_jogador').isNotNull()) \
    .select(
        'partida_id',
        'ano_copa',
        'match_date',
        'nome_jogador',
        'gol_team',
        'tempo_gol',
        'periodo_gol',
        'penalty',
        'owngoal'
    )

# Joins para foreign keys
fato_gols = fato_gols \
    .join(df_copas, on='ano_copa', how='left') \
    .join(df_tempo, on='match_date', how='left') \
    .join(df_jogadores.withColumnRenamed('time_principal', 'gol_team'), on=['nome_jogador', 'gol_team'], how='left') \
    .join(df_times.withColumnRenamed('time_id', 'time_id').withColumnRenamed('time_nome', 'gol_team'), on='gol_team', how='left') \
    .select(
        F.monotonically_increasing_id().alias('gol_id'),
        'partida_id',
        'copa_id',
        'tempo_id',
        'jogador_id',
        'time_id',
        'tempo_gol',
        'periodo_gol',
        'penalty',
        'owngoal'
    )

print(f"  {fato_gols.count()} gols")

fato_gols.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.fato_gols")

print("Tabela workspace.gold.fato_gols criada!")

# Validacao do modelo dimensional
print("\n" + "="*80)
print("VALIDACAO DO MODELO DIMENSIONAL")
print("="*80)

print("\nEstatisticas das Dimensoes:")
print(f"  dim_copas: {spark.table('workspace.gold.dim_copas').count()} registros")
print(f"  dim_times: {spark.table('workspace.gold.dim_times').count()} registros")
print(f"  dim_estadios: {spark.table('workspace.gold.dim_estadios').count()} registros")
print(f"  dim_jogadores: {spark.table('workspace.gold.dim_jogadores').count()} registros")
print(f"  dim_tempo: {spark.table('workspace.gold.dim_tempo').count()} registros")

print("\nEstatisticas das Fatos:")
print(f"  fato_partidas: {spark.table('workspace.gold.fato_partidas').count()} registros")
print(f"  fato_gols: {spark.table('workspace.gold.fato_gols').count()} registros")

print("\nListando todas as tabelas Gold:")
spark.sql("SHOW TABLES IN workspace.gold").show(truncate=False)

print("\nPreview dim_copas:")
spark.sql("SELECT * FROM workspace.gold.dim_copas ORDER BY ano_copa LIMIT 5").show(truncate=False)

print("\nPreview dim_times (primeiros 10):")
spark.sql("SELECT * FROM workspace.gold.dim_times ORDER BY time_nome LIMIT 10").show(truncate=False)

print("\nPreview fato_partidas (primeiros 5):")
spark.sql("""
    SELECT 
        f.partida_id,
        c.copa_nome,
        t1.time_nome as time_casa,
        t2.time_nome as time_visitante,
        f.score_ft_home,
        f.score_ft_away,
        f.resultado_partida
    FROM workspace.gold.fato_partidas f
    JOIN workspace.gold.dim_copas c ON f.copa_id = c.copa_id
    JOIN workspace.gold.dim_times t1 ON f.time_casa_id = t1.time_id
    JOIN workspace.gold.dim_times t2 ON f.time_visitante_id = t2.time_id
    ORDER BY c.ano_copa DESC
    LIMIT 5
""").show(truncate=False)

print("\nPreview fato_gols (primeiros 5):")
spark.sql("""
    SELECT 
        g.gol_id,
        c.copa_nome,
        j.nome_jogador,
        t.time_nome,
        g.tempo_gol,
        g.periodo_gol,
        g.penalty,
        g.owngoal
    FROM workspace.gold.fato_gols g
    JOIN workspace.gold.dim_copas c ON g.copa_id = c.copa_id
    JOIN workspace.gold.dim_jogadores j ON g.jogador_id = j.jogador_id
    JOIN workspace.gold.dim_times t ON g.time_id = t.time_id
    ORDER BY c.ano_copa DESC, g.tempo_gol ASC
    LIMIT 5
""").show(truncate=False)

print("\nModelo Dimensional Gold criado com sucesso!")
print("\nEstrutura do Data Warehouse:")
print("  BRONZE: Dados brutos (matches_json, matches_goals)")
print("  SILVER: Dados transformados e enriquecidos (matches_silver)")
print("  GOLD: Modelo dimensional (5 dimensoes + 2 fatos)")
print("\nProximos passos:")
print("  1. Criar views analiticas agregadas")
print("  2. Criar dashboards e visualizacoes")
print("  3. Implementar analises avancadas")